# Enriquecer mi historial de Spotify con géneros y musicality

**TP Visualización de Datos — ITBA Módulo 4 · "¿Elijo o me eligen?"**

Este notebook toma el export de Spotify y le agrega **géneros por artista** y
**variables de musicality** (BPM, energía, ánimo) desde fuentes externas.

---
### Antes de empezar: dos claves gratuitas (10 minutos en total)

**1 · Spotify** → https://developer.spotify.com/dashboard
Log in con tu cuenta normal → *Create app* → nombre cualquiera, Redirect URI
`http://localhost:8080` → tildá "Web API" → Save. Entrá a la app → *Settings* →
copiá **Client ID** y **Client Secret**.

**2 · Last.fm** → https://www.last.fm/api/account/create
Completá nombre y descripción (cualquier cosa) → te da una **API key** al instante.

> ⚠️ **Importante**: el endpoint `audio-features` de Spotify (danceability, energy,
> valence) **fue dado de baja en noviembre de 2024** para las apps nuevas: devuelve
> 403. Por eso la musicality se busca en otras fuentes, y por eso la celda 7 mide
> cuánta cobertura conseguimos antes de correr todo. Esa limitación va documentada
> en el sitio: es parte del método, no un error.

In [ ]:
# ============================================================
#  1 · CONFIGURACIÓN
# ============================================================
SPOTIFY_CLIENT_ID     = ""   # <-- pegá acá tu Client ID
SPOTIFY_CLIENT_SECRET = ""   # <-- pegá acá tu Client Secret
LASTFM_API_KEY        = ""   # <-- pegá acá tu API key de Last.fm

# Carpeta con el export. En Colab lo más cómodo es subir el ZIP y descomprimirlo.
CARPETA_HISTORIAL = "Spotify Extended Streaming History"
CARPETA_CUENTA    = "Spotify Account Data"   # opcional
SALIDA            = "salida"

# Cuántos artistas enriquecer. Los 4.740 tardan ~20 min por Last.fm.
# Poné un número (ej. 1500) para probar rápido, o None para todos.
LIMITE_ARTISTAS = None

# Cuántos tracks pedirle musicality (los más escuchados primero).
LIMITE_TRACKS_MUSICALITY = 3000

import os, json, time, glob, base64, unicodedata
from collections import Counter
import requests, pandas as pd

os.makedirs(SALIDA, exist_ok=True)
os.makedirs(f"{SALIDA}/cache", exist_ok=True)
print("configurado ✓")

## 2 · Consolidar el historial
Junta los 18 JSON, saca podcasts, pasa a hora argentina y aplica el criterio de
agencia. Es el mismo criterio que se documenta en el sitio.

In [ ]:
ELECCION = {"clickrow", "playbtn", "backbtn", "uriopen", "remote", "popup"}
INERCIA  = {"trackdone", "appload"}

rows = []
for f in sorted(glob.glob(os.path.join(CARPETA_HISTORIAL, "*.json"))):
    with open(f, encoding="utf-8") as fh:
        rows.extend(json.load(fh))
print(f"registros crudos: {len(rows):,}")

df = pd.DataFrame(rows)
df = df[df["master_metadata_track_name"].notna()].copy()      # fuera podcasts
df["ts"] = pd.to_datetime(df["ts"], utc=True)
df["ts_local"] = df["ts"].dt.tz_convert("America/Argentina/Buenos_Aires")
df["fecha"] = df["ts_local"].dt.date
df["anio"]  = df["ts_local"].dt.year
df["mes"]   = df["ts_local"].dt.strftime("%Y-%m")
df["hora"]  = df["ts_local"].dt.hour
df["min_reproducidos"] = df["ms_played"] / 60000
df = df.rename(columns={"master_metadata_track_name": "track",
                        "master_metadata_album_artist_name": "artista",
                        "master_metadata_album_album_name": "album"})
df["track_id"] = df["spotify_track_uri"].str.replace("spotify:track:", "", regex=False)
df["agencia"] = df["reason_start"].map(
    lambda r: "eleccion" if r in ELECCION else
              "inercia"  if r in INERCIA  else
              "arrastre_skip" if r == "fwdbtn" else "otro")
# el campo `skipped` del export está roto: usamos reason_end como proxy
df["fue_skip"] = df["reason_end"].eq("fwdbtn")

print(f"reproducciones de música: {len(df):,}")
print(f"rango: {df['fecha'].min()} → {df['fecha'].max()}")
print(f"artistas únicos: {df['artista'].nunique():,}")

In [ ]:
# lista de artistas a enriquecer, con un track de muestra para resolver su ID
artistas = (df.groupby("artista")
              .agg(minutos=("min_reproducidos", "sum"),
                   reproducciones=("track", "size"))
              .sort_values("minutos", ascending=False)
              .reset_index())
rep = (df.sort_values("min_reproducidos", ascending=False)
         .groupby("artista")["track_id"].first())
artistas["track_id_muestra"] = artistas["artista"].map(rep)

if LIMITE_ARTISTAS:
    artistas = artistas.head(LIMITE_ARTISTAS)
cob = artistas["minutos"].sum() / df["min_reproducidos"].sum() * 100
print(f"a enriquecer: {len(artistas):,} artistas — cubren el {cob:.1f}% de tus minutos")

## 3 · Spotify: los géneros que el algoritmo le pone a cada artista
Dos pasos: `track_id → artist_id` y después `artist_id → genres`. Van de a 50 por
request, así que son unos pocos minutos.

Son **los géneros que usa el propio Spotify**, no una clasificación externa. Para
un TP sobre "¿elijo o me eligen?" esa distinción importa: es la taxonomía del
sistema que te recomienda.

In [ ]:
def token_spotify(cid, secret):
    r = requests.post("https://accounts.spotify.com/api/token",
        headers={"Authorization": "Basic " + base64.b64encode(
                    f"{cid}:{secret}".encode()).decode()},
        data={"grant_type": "client_credentials"}, timeout=20)
    r.raise_for_status()
    return r.json()["access_token"]

TOKEN = token_spotify(SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET)
H = {"Authorization": f"Bearer {TOKEN}"}
print("token de Spotify obtenido ✓")

def spoty(url, params, reintentos=5):
    """GET con manejo de 429 (rate limit) y refresh de token."""
    global TOKEN, H
    for i in range(reintentos):
        r = requests.get(url, headers=H, params=params, timeout=25)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            espera = int(r.headers.get("Retry-After", 2)) + 1
            print(f"  rate limit, espero {espera}s"); time.sleep(espera); continue
        if r.status_code == 401:
            TOKEN = token_spotify(SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET)
            H = {"Authorization": f"Bearer {TOKEN}"}; continue
        if r.status_code in (403, 404):
            return None
        time.sleep(2 ** i)
    return None

def en_lotes(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

# --- paso 1: track_id → artist_id ---
ids = [t for t in artistas["track_id_muestra"].dropna().unique()]
mapa_track_artista = {}
for k, lote in enumerate(en_lotes(ids, 50)):
    d = spoty("https://api.spotify.com/v1/tracks", {"ids": ",".join(lote)})
    if d:
        for tid, tr in zip(lote, d.get("tracks", [])):
            if tr and tr.get("album", {}).get("artists"):
                a = tr["album"]["artists"][0]
                mapa_track_artista[tid] = (a["id"], a["name"])
    if k % 20 == 0:
        print(f"  tracks resueltos: {len(mapa_track_artista):,}/{len(ids):,}")
    time.sleep(0.08)

artistas["artist_id"] = artistas["track_id_muestra"].map(
    lambda t: mapa_track_artista.get(t, (None, None))[0])
print(f"artist_id resuelto para {artistas['artist_id'].notna().sum():,} "
      f"de {len(artistas):,} artistas")

In [ ]:
# --- paso 2: artist_id → genres, popularity, followers ---
aids = artistas["artist_id"].dropna().unique().tolist()
info = {}
for k, lote in enumerate(en_lotes(aids, 50)):
    d = spoty("https://api.spotify.com/v1/artists", {"ids": ",".join(lote)})
    if d:
        for a in d.get("artists", []):
            if a:
                info[a["id"]] = {
                    "generos_spotify": a.get("genres", []),
                    "popularidad": a.get("popularity"),
                    "seguidores": (a.get("followers") or {}).get("total"),
                }
    if k % 20 == 0:
        print(f"  artistas resueltos: {len(info):,}/{len(aids):,}")
    time.sleep(0.08)

artistas["generos_spotify"] = artistas["artist_id"].map(
    lambda i: info.get(i, {}).get("generos_spotify", []))
artistas["popularidad"] = artistas["artist_id"].map(
    lambda i: info.get(i, {}).get("popularidad"))
artistas["seguidores"] = artistas["artist_id"].map(
    lambda i: info.get(i, {}).get("seguidores"))

con_gen = artistas["generos_spotify"].apply(len).gt(0)
print(f"\nSpotify devolvió géneros para {con_gen.sum():,} de {len(artistas):,} "
      f"artistas ({con_gen.mean()*100:.1f}%)")
print("→ Last.fm (celda siguiente) cubre el resto.")
artistas.to_json(f"{SALIDA}/cache/artistas_spotify.json", orient="records")

## 4 · Last.fm: tags de la comunidad
Un request por artista, ~5 por segundo. Rellena lo que Spotify deja vacío y aporta
etiquetas que Spotify no tiene ("rock nacional", "boom bap", "trova").
Guarda cache: si se corta, volvé a correr la celda y sigue donde quedó.

In [ ]:
CACHE_LFM = f"{SALIDA}/cache/lastfm.json"
cache = json.load(open(CACHE_LFM)) if os.path.exists(CACHE_LFM) else {}

def tags_lastfm(nombre):
    if nombre in cache:
        return cache[nombre]
    try:
        r = requests.get("https://ws.audioscrobbler.com/2.0/", timeout=20, params={
            "method": "artist.getTopTags", "artist": nombre,
            "api_key": LASTFM_API_KEY, "format": "json", "autocorrect": 1})
        tags = [t["name"].lower() for t in r.json().get("toptags", {}).get("tag", [])
                if int(t.get("count", 0)) >= 15][:8]
    except Exception:
        tags = []
    cache[nombre] = tags
    return tags

nombres = artistas["artista"].tolist()
for i, n in enumerate(nombres):
    tags_lastfm(n)
    time.sleep(0.21)
    if i % 250 == 0:
        json.dump(cache, open(CACHE_LFM, "w"))
        print(f"  {i:,}/{len(nombres):,} — último: {n[:40]}")
json.dump(cache, open(CACHE_LFM, "w"))

artistas["tags_lastfm"] = artistas["artista"].map(lambda n: cache.get(n, []))
con_tags = artistas["tags_lastfm"].apply(len).gt(0)
print(f"\nLast.fm devolvió tags para {con_tags.sum():,} "
      f"({con_tags.mean()*100:.1f}%)")

cualquiera = con_gen | con_tags
print(f"COBERTURA COMBINADA: {cualquiera.sum():,} de {len(artistas):,} "
      f"({cualquiera.mean()*100:.1f}%)")
peso = artistas.loc[cualquiera, "minutos"].sum() / artistas["minutos"].sum() * 100
print(f"…que representan el {peso:.1f}% de tus minutos escuchados")

## 5 · Agrupar en macro-géneros
Spotify y Last.fm devuelven etiquetas muy finas ("conscious hip hop", "neo soul",
"argentine rock"). Para un evolutivo de 12 años eso es ilegible: hay que agrupar.
El mapeo de abajo es **una decisión tuya y hay que defenderla en la oral** —
cambialo si no te representa.

In [ ]:
MACRO = {
    "Hip-hop / Rap":  ["hip hop", "hip-hop", "rap", "trap", "boom bap", "drill",
                       "grime", "hiphop"],
    "R&B / Soul":     ["r&b", "rnb", "soul", "neo soul", "funk", "motown"],
    "Rock nacional":  ["argentine", "rock nacional", "argentin", "rock argentino",
                       "uruguay"],
    "Rock / Metal":   ["rock", "metal", "punk", "grunge", "hardcore", "britpop",
                       "shoegaze"],
    "Electrónica":    ["electronic", "house", "techno", "edm", "idm", "dubstep",
                       "drum and bass", "downtempo", "trip hop", "ambient"],
    "Jazz / Blues":   ["jazz", "blues", "bebop", "swing"],
    "Latino / Urbano":["reggaeton", "latin", "cumbia", "salsa", "bachata",
                       "merengue", "urbano", "perreo"],
    "Trova / Folk":   ["folk", "trova", "cantautor", "singer-songwriter",
                       "nueva cancion", "flamenco", "tango", "chanson"],
    "Pop":            ["pop"],
    "Reggae":         ["reggae", "dub", "ska", "dancehall"],
    "Indie / Alt":    ["indie", "alternative", "lo-fi", "lofi", "dream pop"],
    "Clásica / BSO":  ["classical", "soundtrack", "score", "opera", "piano"],
}
ORDEN = ["Rock nacional", "Trova / Folk", "Latino / Urbano", "Reggae",
         "Hip-hop / Rap", "R&B / Soul", "Jazz / Blues", "Electrónica",
         "Indie / Alt", "Rock / Metal", "Pop", "Clásica / BSO"]

def a_macro(etiquetas):
    """Primer match según ORDEN: lo específico gana sobre lo genérico."""
    txt = " | ".join(etiquetas).lower()
    for macro in ORDEN:
        if any(p in txt for p in MACRO[macro]):
            return macro
    return "Otros"

artistas["etiquetas"] = artistas.apply(
    lambda r: list(r["generos_spotify"]) + list(r["tags_lastfm"]), axis=1)
artistas["macro_genero"] = artistas["etiquetas"].apply(a_macro)
artistas["genero_detalle"] = artistas["etiquetas"].apply(
    lambda e: e[0] if e else "sin dato")

print(artistas.groupby("macro_genero")["minutos"].sum().div(60).round(0)
        .sort_values(ascending=False).rename("horas").to_string())
sin = artistas["macro_genero"].eq("Otros")
print(f"\nsin clasificar: {sin.sum():,} artistas "
      f"({artistas.loc[sin,'minutos'].sum()/artistas['minutos'].sum()*100:.1f}% "
      f"de los minutos)")
print("\ntop 15 sin clasificar (revisá si merecen una regla nueva):")
print(artistas[sin].nlargest(15, "minutos")[["artista", "minutos", "etiquetas"]]
        .to_string(index=False))

artistas.to_csv(f"{SALIDA}/artistas_enriquecidos.csv", index=False, encoding="utf-8")

## 6 · Los CSV del evolutivo de géneros
Tres cortes: streamgraph por mes, participación por año y el "año de descubrimiento"
de cada macro-género.

In [ ]:
g = df.merge(artistas[["artista", "macro_genero", "genero_detalle",
                       "popularidad"]], on="artista", how="left")
g["macro_genero"] = g["macro_genero"].fillna("Otros")

# V6a — streamgraph mensual (horas por género)
(g.pivot_table(index="mes", columns="macro_genero",
               values="min_reproducidos", aggfunc="sum")
   .fillna(0).div(60).round(2).reset_index()
   .to_csv(f"{SALIDA}/v6_generos_por_mes.csv", index=False, encoding="utf-8"))

# V6b — participación porcentual por año
anual = (g.pivot_table(index="anio", columns="macro_genero",
                       values="min_reproducidos", aggfunc="sum").fillna(0))
(anual.div(anual.sum(axis=1), axis=0).mul(100).round(2).reset_index()
   .to_csv(f"{SALIDA}/v6_generos_por_anio_pct.csv", index=False, encoding="utf-8"))

# V6c — cuándo apareció cada género y cuándo fue su pico
hitos = (g.groupby("macro_genero")
           .agg(primer_play=("fecha", "min"), horas=("min_reproducidos", "sum"),
                artistas=("artista", "nunique"))
           .assign(horas=lambda d: (d.horas / 60).round(1)))
hitos["anio_pico"] = anual.idxmax()
hitos.reset_index().to_csv(f"{SALIDA}/v6_hitos_generos.csv",
                           index=False, encoding="utf-8")
print(hitos.sort_values("horas", ascending=False).to_string())

# V6d — ¿elijo unos géneros y otros me caen encima?
ag = (g.pivot_table(index="macro_genero", columns="agencia",
                    values="min_reproducidos", aggfunc="sum").fillna(0))
ag = ag.div(ag.sum(axis=1), axis=0).mul(100).round(1)
ag = ag.join(g.groupby("macro_genero")["min_reproducidos"].sum().div(60)
               .round(0).rename("horas"))
ag.sort_values("eleccion", ascending=False).reset_index().to_csv(
    f"{SALIDA}/v6_agencia_por_genero.csv", index=False, encoding="utf-8")
print("\n% de elección deliberada por género — el número más interesante del TP:")
print(ag.sort_values("eleccion", ascending=False)[["eleccion", "horas"]].to_string())

## 7 · Musicality: primero medimos si vale la pena
Spotify dio de baja `audio-features`. Esta celda **prueba 3 fuentes con 15 tracks**
y te dice cuál funciona y con cuánta cobertura, antes de lanzar miles de requests.
Si ninguna supera ~40%, lo honesto es dejar la musicality afuera y decirlo en el
sitio — un dato con la mitad de huecos no se puede defender en la oral.

In [ ]:
muestra = (df.groupby("track_id")["min_reproducidos"].sum()
             .nlargest(15).index.tolist())
nombres_muestra = (df.drop_duplicates("track_id").set_index("track_id")
                     .loc[muestra, ["track", "artista"]])

def probar_reccobeats(ids):
    ok = 0
    for tid in ids:
        try:
            r = requests.get("https://api.reccobeats.com/v1/track",
                             params={"ids": tid}, timeout=15)
            if r.status_code == 200 and r.json().get("content"):
                rid = r.json()["content"][0]["id"]
                a = requests.get(
                    f"https://api.reccobeats.com/v1/track/{rid}/audio-features",
                    timeout=15)
                if a.status_code == 200 and a.json():
                    ok += 1
        except Exception:
            pass
        time.sleep(0.3)
    return ok

def probar_deezer(filas):
    ok = 0
    for _, row in filas.iterrows():
        try:
            q = f'track:"{row["track"]}" artist:"{row["artista"]}"'
            r = requests.get("https://api.deezer.com/search",
                             params={"q": q}, timeout=15).json()
            if r.get("data"):
                t = requests.get(
                    f"https://api.deezer.com/track/{r['data'][0]['id']}",
                    timeout=15).json()
                if t.get("bpm"):
                    ok += 1
        except Exception:
            pass
        time.sleep(0.3)
    return ok

def probar_spotify_af(ids):
    r = requests.get("https://api.spotify.com/v1/audio-features",
                     headers=H, params={"ids": ",".join(ids)}, timeout=20)
    return r.status_code

print("sondeando fuentes de musicality…\n")
cod = probar_spotify_af(muestra[:5])
print(f"  Spotify audio-features : HTTP {cod} "
      f"{'(dado de baja, como esperábamos)' if cod in (403,404,410) else '(¡anda!)'}")
rb = probar_reccobeats(muestra)
print(f"  ReccoBeats             : {rb}/15 ({rb/15*100:.0f}%)")
dz = probar_deezer(nombres_muestra)
print(f"  Deezer (BPM)           : {dz}/15 ({dz/15*100:.0f}%)")

mejor = max([("reccobeats", rb), ("deezer", dz)], key=lambda x: x[1])
print(f"\n→ mejor fuente: {mejor[0]} con {mejor[1]/15*100:.0f}% de cobertura")
if mejor[1] / 15 < 0.4:
    print("→ RECOMENDACIÓN: saltear la celda 8. Documentá en el sitio que la "
          "musicality no era recuperable tras la baja del endpoint de Spotify. "
          "Eso es un hallazgo válido del TP, no una carencia.")
else:
    print("→ dale a la celda 8.")

## 8 · Musicality masiva *(sólo si la celda 7 dio buena cobertura)*
Corre sobre los tracks más escuchados. Guarda cache cada 100: si Colab te corta,
volvé a ejecutar y sigue donde quedó.

In [ ]:
FUENTE = mejor[0]        # o forzalo a mano: "reccobeats" / "deezer"

top_tracks = (df.groupby("track_id")
                .agg(minutos=("min_reproducidos", "sum"),
                     track=("track", "first"), artista=("artista", "first"))
                .nlargest(LIMITE_TRACKS_MUSICALITY, "minutos").reset_index())

CACHE_MUS = f"{SALIDA}/cache/musicality.json"
mus = json.load(open(CACHE_MUS)) if os.path.exists(CACHE_MUS) else {}

def features_reccobeats(tid):
    try:
        r = requests.get("https://api.reccobeats.com/v1/track",
                         params={"ids": tid}, timeout=15)
        c = r.json().get("content") if r.status_code == 200 else None
        if not c: return None
        a = requests.get(
            f"https://api.reccobeats.com/v1/track/{c[0]['id']}/audio-features",
            timeout=15)
        return a.json() if a.status_code == 200 else None
    except Exception:
        return None

def features_deezer(track, artista):
    try:
        q = f'track:"{track}" artist:"{artista}"'
        r = requests.get("https://api.deezer.com/search",
                         params={"q": q}, timeout=15).json()
        if not r.get("data"): return None
        t = requests.get(f"https://api.deezer.com/track/{r['data'][0]['id']}",
                         timeout=15).json()
        return {"tempo": t.get("bpm"), "loudness": t.get("gain"),
                "duracion_s": t.get("duration"), "rank_deezer": t.get("rank")}
    except Exception:
        return None

for i, row in top_tracks.iterrows():
    tid = row["track_id"]
    if tid in mus:
        continue
    mus[tid] = (features_reccobeats(tid) if FUENTE == "reccobeats"
                else features_deezer(row["track"], row["artista"])) or {}
    time.sleep(0.25)
    if i % 100 == 0:
        json.dump(mus, open(CACHE_MUS, "w"))
        hit = sum(1 for v in mus.values() if v)
        print(f"  {i:,}/{len(top_tracks):,} — con datos: {hit:,} "
              f"({hit/max(len(mus),1)*100:.0f}%)")
json.dump(mus, open(CACHE_MUS, "w"))

mdf = pd.DataFrame([{"track_id": k, **v} for k, v in mus.items() if v])
if len(mdf):
    mdf = top_tracks.merge(mdf, on="track_id", how="left")
    mdf.to_csv(f"{SALIDA}/musicality_tracks.csv", index=False, encoding="utf-8")
    num = [c for c in ("tempo", "energy", "valence", "danceability", "loudness")
           if c in mdf.columns]
    cobertura = mdf[num[0]].notna().mean() * 100 if num else 0
    print(f"\nCOBERTURA FINAL: {cobertura:.1f}% de los "
          f"{len(top_tracks):,} tracks más escuchados")
    print("↑ este número va escrito en el sitio, tal cual.")

    # evolutivo de musicality por año, ponderado por minutos
    if num:
        # promedio ponderado por minutos escuchados, año a año
        j = df.merge(mdf[["track_id"] + num], on="track_id", how="inner")
        filas = []
        for anio, d in j.groupby("anio"):
            fila = {"anio": anio, "tracks_con_dato": d["track_id"].nunique()}
            for c in num:
                v = d[d[c].notna()]
                peso = v["min_reproducidos"].sum()
                fila[c] = round((v[c] * v["min_reproducidos"]).sum() / peso, 3) \
                          if peso else None
            filas.append(fila)
        ev = pd.DataFrame(filas)
        ev.to_csv(f"{SALIDA}/v10_musicality_por_anio.csv",
                  index=False, encoding="utf-8")
        print(ev.to_string(index=False))
else:
    print("Sin datos de musicality. Documentalo en el sitio y seguí: "
          "los géneros ya sostienen el evolutivo.")

## 9 · Listo
En la carpeta `salida/` quedaron:

| archivo | para qué |
|---|---|
| `artistas_enriquecidos.csv` | la base maestra de artistas con géneros |
| `v6_generos_por_mes.csv` | streamgraph de géneros (Flourish) |
| `v6_generos_por_anio_pct.csv` | participación % por año (Datawrapper) |
| `v6_hitos_generos.csv` | cuándo apareció y cuándo pegó cada género |
| `v6_agencia_por_genero.csv` | **% de elección por género** ← el mejor |
| `musicality_tracks.csv` | BPM / energía, si la celda 7 dio verde |
| `v10_musicality_por_anio.csv` | evolutivo de musicality |

En Colab: panel de archivos (📁 a la izquierda) → botón derecho sobre `salida` →
*Descargar*. Después los subís a `/spotify/data/` en el repo.

In [ ]:
import shutil
shutil.make_archive("salida_enriquecida", "zip", SALIDA)
print("listo → salida_enriquecida.zip")
for f in sorted(os.listdir(SALIDA)):
    p = f"{SALIDA}/{f}"
    if os.path.isfile(p):
        print(f"  {f:36s} {os.path.getsize(p)/1024:8.1f} KB")
try:
    from google.colab import files
    files.download("salida_enriquecida.zip")
except Exception:
    pass